# Food Traceability System

This projetct is used to demonstrates how blockchain technology can be used for food traceability.



In [1]:
from web3 import Web3

# to connect the Ganache blockchain
ganache_url = "http://127.0.0.1:7545"

w3 = Web3(Web3.HTTPProvider(ganache_url))

# checking connection
if w3.is_connected():
    print("Successfully connected to Ganache!")
    print("Chain ID:", w3.eth.chain_id)
    print("Latest block:", w3.eth.block_number)
    print("Available accounts:", len(w3.eth.accounts))
else:
    print("Connection to Ganache failed.")

Successfully connected to Ganache!
Chain ID: 1337
Latest block: 3
Available accounts: 10


In [2]:
# Geting accounts from ganashe
accounts = w3.eth.accounts

# Selecting producer and retailer
producer_account = accounts[0]
retailer_account = accounts[1]

print("Producer account:", producer_account)
print("Retailer account:", retailer_account)

Producer account: 0x64D9a9238d75040b1d7150Aa5D9a591a45f61391
Retailer account: 0xfd82b13C62c01211a1Cd6fAa2f1A7D170028CB81


In [5]:
import json


contract_address = "0x3531f4D7Dce9De60499cAE7729cA7227eC11694A"

# Smart contract 
abi_json = r'''[
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": true,
				"internalType": "bytes32",
				"name": "batchId",
				"type": "bytes32"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "name",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "sender",
				"type": "address"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "receiver",
				"type": "address"
			}
		],
		"name": "BatchRegistered",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "batchId",
				"type": "string"
			}
		],
		"name": "confirmReceipt",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": true,
				"internalType": "bytes32",
				"name": "batchId",
				"type": "bytes32"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "receiver",
				"type": "address"
			}
		],
		"name": "ReceiptConfirmed",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "batchId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "productName",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "origin",
				"type": "string"
			},
			{
				"internalType": "address",
				"name": "receiver",
				"type": "address"
			}
		],
		"name": "registerBatch",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "batchId",
				"type": "string"
			}
		],
		"name": "getBatch",
		"outputs": [
			{
				"internalType": "string",
				"name": "productName",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "origin",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "status",
				"type": "string"
			},
			{
				"internalType": "address",
				"name": "sender",
				"type": "address"
			},
			{
				"internalType": "address",
				"name": "receiver",
				"type": "address"
			},
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "bool",
				"name": "received",
				"type": "bool"
			}
		],
		"stateMutability": "view",
		"type": "function"
	}
]'''

contract_abi = json.loads(abi_json)

# Connect to the smart contract
contract = w3.eth.contract(
    address=Web3.to_checksum_address(contract_address),
    abi=contract_abi
)

# Check if the contract exists
contract_code = w3.eth.get_code(contract.address)

if len(contract_code) > 0:
    print("Successfully connected to FoodTraceability!")
    print("Contract address:", contract.address)
else:
    print("Contract not found at this address.")

Successfully connected to FoodTraceability!
Contract address: 0x3531f4D7Dce9De60499cAE7729cA7227eC11694A


In [6]:
#getting the information from the blockchain
batch = contract.functions.getBatch("FOOD001").call()

#diplaying information
print("Food Batch Information")
print("----------------------")
print("Product:", batch[0])
print("Origin:", batch[1])
print("Status:", batch[2])
print("Producer:", batch[3])
print("Retailer:", batch[4])
print("Timestamp:", batch[5])
print("Received:", batch[6])

Food Batch Information
----------------------
Product: Organic Tomatos
Origin: Ireland
Status: Received
Producer: 0x64D9a9238d75040b1d7150Aa5D9a591a45f61391
Retailer: 0xfd82b13C62c01211a1Cd6fAa2f1A7D170028CB81
Timestamp: 1793718312
Received: True


In [ ]:
# Register a new food batch using the producer account

tx_hash = contract.functions.registerBatch(
    "FOOD002",
    "Organic Apples",
    "Ireland",
    retailer_account
).transact({
    "from": producer_account
})

# Wait for the transaction to be confirmed
tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

print("Batch registration completed!")
print("Transaction successful:", tx_receipt.status == 1)
print("Block number:", tx_receipt.blockNumber)
print("Transaction hash:", tx_hash.hex())

In [8]:
# this part registers a new food but using the producer account

tx_hash = contract.functions.registerBatch(
    "FOOD002",
    "Organic Apples",
    "Ireland",
    retailer_account
).transact({
    "from": producer_account
})

#this part is for the transaction confirmation
tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

print("Batch registration completed!")
print("Transaction successful:", tx_receipt.status == 1)
print("Block number:", tx_receipt.blockNumber)
print("Transaction hash:", tx_hash.hex())

Batch registration completed!
Transaction successful: True
Block number: 4
Transaction hash: 59724c6250f69b04570b120951cabcc81a6b837f9a20f7026fcdb779fba12f19


In [9]:
# This part checks the new batch

batch_2 = contract.functions.getBatch("FOOD002").call()

print("Food Batch Information")
print("----------------------")
print("Product:", batch_2[0])
print("Origin:", batch_2[1])
print("Status:", batch_2[2])
print("Producer:", batch_2[3])
print("Retailer:", batch_2[4])
print("Received:", batch_2[6])

Food Batch Information
----------------------
Product: Organic Apples
Origin: Ireland
Status: Harvested
Producer: 0x64D9a9238d75040b1d7150Aa5D9a591a45f61391
Retailer: 0xfd82b13C62c01211a1Cd6fAa2f1A7D170028CB81
Received: False
